# 01 · CPSC Recalls API

**Agency:** U.S. Consumer Product Safety Commission — household products, toys, baby gear, furniture, electronics.
**Endpoint:** `https://www.saferproducts.gov/RestWebServices/Recall`  ·  **Auth:** none  ·  **Format:** XML by default, add `format=json`

**Questions this notebook answers**
1. What does one recall look like?
2. Which fields hold the model number, lot/date code and UPC — and how often are they filled?
3. When they're not in a field, where are they hiding?

> Saved samples: 20 recalls chosen to match our use cases (baby gear, kids' toys, fire hazards, tip-overs) plus 2 older recalls that have UPCs.

In [1]:
import sys, json, re
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # so we can import recall_explore from the repo root
import pandas as pd
from recall_explore import fetch, load_samples, find_ids, coverage, normalize_model
pd.set_option("display.max_colwidth", 80)

LIVE = False   # True = call the real API (needs internet). False = use the 20 saved samples.

## 1 · Fetch
The API filters by query parameters such as `RecallDateStart`, `RecallDateEnd`, `RecallNumber`, `ProductName`, `Title`.
There is **no paging** — a date window returns every recall in it, so for the backfill we'll walk month by month.

In [2]:
if LIVE:
    recalls = fetch("cpsc", RecallDateStart="2026-09-20", RecallDateEnd="2026-10-20")
    print(len(recalls), "recalls in window")
else:
    recalls = load_samples("cpsc")
print(len(recalls), "records loaded")

20 records loaded


## 2 · Anatomy of one recall
Top-level keys, then the nested lists. Notice most interesting things are **lists of objects** (`Products`, `Hazards`, `Remedies` …), not plain strings.

In [3]:
r = next(x for x in recalls if x["RecallNumber"] == "27027")   # Papablic bedside sleeper (baby gear)
for k, v in r.items():
    if k.startswith("_"): continue
    print(f"{k:<22}", json.dumps(v, ensure_ascii=False)[:110])

RecallID               11024
RecallNumber           "27027"
RecallDate             "2026-10-15T00:00:00"
Description            "This recall involves Papablic bedside sleepers, model PAPA03001AI. These convertible sleepers function as bot
URL                    "https://cpsc.gov/Recalls/2027/Papablic-of-USA-Recalls-Convertible-Bedside-Sleepers-Due-to-Risk-of-Serious-Inj
Title                  "Papablic, of USA Recalls Convertible Bedside Sleepers Due to Risk of Serious Injury or Death from Entrapment,
ConsumerContact        "Papablic by email at support@papablic.com, or online at https://papablic.com/pages/black-and-gray-bassinet-wi
LastPublishDate        "2026-10-15T00:00:00"
Products               [{"Name": "Papablic Bedside Sleepers with Play Yard Extensions", "Description": "", "Model": "", "Type": "Play
Inconjunctions         []
Images                 [{"URL": "https://www.cpsc.gov/s3fs-public/Papablic1.png", "Caption": "Recalled Papablic Convertible Bedside S
Injuries            

## 3 · Field coverage
How often is each field actually filled? This is the single most important table in this notebook.

In [4]:
flat = []
for x in recalls:
    p = x["Products"][0] if x["Products"] else {}
    flat.append({**x,
        "Products[].Model": p.get("Model"),
        "Products[].Name": p.get("Name"),
        "Products[].NumberOfUnits": p.get("NumberOfUnits"),
    })
coverage(flat, ["Title", "Description", "Products[].Name", "Products[].Model", "ProductUPCs",
                "Products[].NumberOfUnits", "Manufacturers", "Importers", "Retailers",
                "Hazards", "Remedies", "Images", "Injuries"])

,filled,of,pct
field,,,
Title,20,20,100
Description,20,20,100
Products[].Name,20,20,100
Products[].Model,0,20,0
ProductUPCs,2,20,10
Products[].NumberOfUnits,20,20,100
Manufacturers,3,20,15
Importers,13,20,65
Retailers,20,20,100


**Read this carefully.** `Products[].Model` is **0 %** filled. The field exists, but CPSC leaves it blank.
We double-checked this across **93 recalls from three periods** (Sep–Oct 2026, Mar 2024, Jun 2019) during the raw pull: the Model field was empty in every one, and only 2 had `ProductUPCs` (both from 2019).

➡️ **For CPSC, identifiers live in the `Description` prose.** That's why our pipeline needs an extraction step.

## 4 · Where the identifiers hide: regex first pass
`find_ids()` (in `recall_explore.py`) uses simple patterns like *"model number X"*, *"UPC 1234…"*, *"Lot: X"*. Whatever it misses is the work an LLM has to do.

In [5]:
rows = []
for x in recalls:
    ids = find_ids(x["Description"])
    rows.append({"recall": x["RecallNumber"], "tag": x.get("_sample_tag"),
                 "product": x["Products"][0]["Name"][:45] if x["Products"] else "",
                 "models_found": ", ".join(ids["models"]) or "—",
                 "upcs_in_text": len(ids["upcs"]),
                 "upcs_in_field": len(x["ProductUPCs"])})
found = pd.DataFrame(rows).set_index("recall")
found

,tag,product,models_found,upcs_in_text,upcs_in_field
recall,,,,,
19131,has_upc,Small Stratton glass pendant light fixtures,—,0,16
19154,has_upc,Venus Simply3 Disposable razors,—,2,2
26789,kids,5Color Children’s Bike Helmet and Pads Sets,YD-001,0,0
26791,fire,The Blue Cactus Company 2500 mAh Battery Pack,RWX-RBP02,0,0
26792,fire,NEWDERY power banks,ZHX-PB22,0,0
26793,fire,WinxMatrs Mattresses,—,0,0
26798,fire,Portable Sleep Machines with Night Light mode,LTD-SM23,0,0
27003,kids,"Squishy Dumpling Toys, Crazy Squib Ball Toys","3655, 3984",0,0
27005,bedrail,LandTale Adult Portable Bed Rails,EBR001,0,0


In [6]:
has_model = (found.models_found != "—").sum()
print(f"Regex found a model number in {has_model}/{len(found)} descriptions")
print(f"Structured Model field filled in 0/{len(found)}")

Regex found a model number in 13/20 descriptions
Structured Model field filled in 0/20


## 5 · The three kinds of CPSC description
Looking at the misses, every recall falls into one of three buckets — each needs a different matching strategy:

In [7]:
def bucket(x):
    d = x["Description"]
    ids = find_ids(d)
    if re.search(r"(model|item)\s*(no\.?|numbers?)[^.]{0,40}(included|following|:)", d, re.I) and len(re.findall(r"\b[A-Z0-9]{8,}\b", d)) >= 3:
        return "B · table flattened into prose"
    if ids["models"] or ids["upcs"]:
        return "A · single model in a sentence"
    return "C · no identifier at all"
pd.Series({x["RecallNumber"]: bucket(x) for x in recalls}).value_counts()

A · single model in a sentence    14
C · no identifier at all           4
B · table flattened into prose     2
Name: count, dtype: int64

| Bucket | Example | What it means for matching |
|---|---|---|
| **A** single model in a sentence | 27024 *"…thermal laminator, model number TL1302Z"* | Regex/LLM extracts it → exact match on `model_norm` |
| **B** table flattened into prose | 27010 grill brushes, 19131 pendant lights: `Item Number UPC 700TDSTNPSCB-PAR 884655362979 …` | Table structure was lost; an LLM must re-pair model ↔ UPC ↔ date range |
| **C** no identifier | 27020 dresser, 26793 mattress, 27022 medal | Only brand + product name + photo. Needs **semantic match** (embeddings) and the image, and should be shown as *Possible match* |

In [8]:
for n in ["27010", "27020"]:
    x = next(y for y in recalls if y["RecallNumber"] == n)
    print(n, "→", x["Description"][:420], "\n")

27010 → This recall involves certain Walmart Expert Grill metal wire bristle grill brushes. The brushes have black plastic or stainless-steel handles. Some brushes come with metal scraper attached. The following model numbers are included in this recall and can be found on the product packaging or a sticker/tag attached to the brush: Model No. Brush Description Distribution Dates EX5607031616002 Expert Grill 16-inch Steel Fi 

27020 → This recall involves Romorgniz six-drawer dressers. The wooden dressers were sold in white and black, measure about 27.6 inches wide, 15.6 inches long, 43.2 inches tall and weigh 73.3 pounds.  

 



## 6 · Other things worth knowing
- **Dates** are ISO strings with a fake time: `2026-10-15T00:00:00`. Parse to `date`.
- `LastPublishDate` changes when a recall is updated → use it to detect updates on re-ingest.
- `Retailers[].Name` often contains *where and when it sold and for how much* — useful for the UI ("Sold on Amazon, 2024–2026, $44–$56").
- `Images[]` has real product photos with captions → great for the results page and for a future image-similarity match.
- Text contains Unicode (™, curly quotes, `\u2011`) → normalise before matching.
- One `RecallNumber` = one announcement, but it can cover **many models** → confirms the `recalls` 1→* `recall_products` table design.

In [9]:
x = next(y for y in recalls if y["RecallNumber"] == "27024")
print("Retailer:", x["Retailers"][0]["Name"])
print("Image   :", x["Images"][0]["URL"], "-", x["Images"][0]["Caption"])
print("Units   :", x["Products"][0]["NumberOfUnits"])
print("Remedy  :", x["RemedyOptions"])

Retailer: Sold Online At:
Amazon.com from June 2024 through September 2026 for between $44 and $56.
Image   : https://www.cpsc.gov/s3fs-public/3M1.png - Recalled Scotch™ Thermal Laminator model number TL1302Z
Units   : About 53,634
Remedy  : [{'Option': 'Refund'}]


## ✅ Takeaways for Recall Radar
1. **Never trust `Products[].Model` or `ProductUPCs`** — extract from `Description` instead.
2. In our 20 samples: 14 name a model/UPC in a sentence (easy), 2 hide a whole table in prose (LLM), and **4 (1 in 5) have no identifier at all** (semantic + photo).
3. Keep the raw JSON (`recalls.raw`) — we'll re-extract when prompts improve.
4. Use `Images`, `Retailers`, `NumberOfUnits` in the UI; they make the result feel trustworthy.